In [15]:
!pip install numpy scikit-learn xgboost lightgbm shap matplotlib

You should consider upgrading via the '/Users/krishnareddy/ my Projects/credlytic/backend/.venv/bin/python3 -m pip install --upgrade pip' command.


In [1]:
# ============================================================
# CELL 1: Imports and Load Data
# ============================================================
import pandas as pd
import numpy as np
import os

# Build path relative to this notebook's location
BASE_DIR = os.path.dirname(os.getcwd())
if os.path.basename(os.getcwd()) != "ml":
    BASE_DIR = os.getcwd()

data_path = os.path.join(BASE_DIR, "data", "datasets", "crx.data")
df = pd.read_csv(data_path, header=None)

print("Shape:", df.shape)
print(df.head())

Shape: (690, 16)
  0      1      2  3  4  5  6     7  8  9   10 11 12     13   14 15
0  b  30.83  0.000  u  g  w  v  1.25  t  t   1  f  g  00202    0  +
1  a  58.67  4.460  u  g  q  h  3.04  t  t   6  f  g  00043  560  +
2  a  24.50  0.500  u  g  q  h  1.50  t  f   0  f  g  00280  824  +
3  b  27.83  1.540  u  g  w  v  3.75  t  t   5  t  g  00100    3  +
4  b  20.17  5.625  u  g  w  v  1.71  t  f   0  f  s  00120    0  +


In [2]:
# ============================================================
# CELL 2: Clean missing values and rename columns
# ============================================================

# Replace '?' with NaN
df = df.replace('?', np.nan)

# Check missing values
print("Missing values per column:")
print(df.isnull().sum())

# Rename columns to meaningful names
df.columns = [
    'gender', 'age', 'debt', 'married', 'bank_customer',
    'education', 'ethnicity', 'years_employed', 'prior_default',
    'employed', 'credit_score', 'drivers_license', 'citizen',
    'zip_code', 'income', 'approved'
]

# Drop rows with missing values (only ~5% of data)
df = df.dropna()
print(f"\nRows after dropping NaN: {df.shape[0]}")

# Convert target column: + -> 1 (approved), - -> 0 (denied)
df['approved'] = df['approved'].map({'+': 1, '-': 0})

# Fix dtypes
df['age'] = df['age'].astype(float)
df['zip_code'] = df['zip_code'].astype(int)

print(f"\nApproved: {int(df['approved'].sum())}, Denied: {int((df['approved'] == 0).sum())}")
print(df.head())

Missing values per column:
0     12
1     12
2      0
3      6
4      6
5      9
6      9
7      0
8      0
9      0
10     0
11     0
12     0
13    13
14     0
15     0
dtype: int64

Rows after dropping NaN: 653

Approved: 296, Denied: 357
  gender    age   debt married  ... citizen zip_code income  approved
0      b  30.83  0.000       u  ...       g      202      0         1
1      a  58.67  4.460       u  ...       g       43    560         1
2      a  24.50  0.500       u  ...       g      280    824         1
3      b  27.83  1.540       u  ...       g      100      3         1
4      b  20.17  5.625       u  ...       s      120      0         1

[5 rows x 16 columns]


In [3]:
# ============================================================
# CELL 3: Generate Synthetic Indian Banking Features
# ============================================================
np.random.seed(42)
n = len(df)

# CIBIL Score (300-900), correlated with approval
approved_mask = df['approved'].values == 1
cibil = np.where(
    approved_mask,
    np.random.randint(650, 900, n),
    np.random.randint(300, 700, n)
)
df['cibil_score'] = cibil

# Monthly income in INR (salary bands)
income_approved = np.array([25000, 35000, 50000, 75000, 100000, 150000])
income_denied = np.array([12000, 18000, 25000, 30000, 40000])
monthly_inc = np.where(
    approved_mask,
    np.random.choice(income_approved, n),
    np.random.choice(income_denied, n)
)
df['monthly_income'] = monthly_inc

# Existing EMI amount (0-40% of income)
df['existing_emi'] = (df['monthly_income'].values * np.random.uniform(0.0, 0.4, n)).astype(int)

# Employment type
emp_types = np.array(['salaried', 'self_employed', 'business', 'freelancer'])
df['employment_type'] = np.random.choice(emp_types, n, p=[0.5, 0.2, 0.2, 0.1])

# City tier
tiers = np.array(['tier_1', 'tier_2', 'tier_3'])
df['city_tier'] = np.random.choice(tiers, n, p=[0.4, 0.35, 0.25])

print("Indian banking features added:")
print(df[['cibil_score', 'monthly_income', 'existing_emi', 'employment_type', 'city_tier', 'approved']].head(10))

Indian banking features added:
   cibil_score  monthly_income  ...  city_tier approved
0          752           25000  ...     tier_1        1
1          829           25000  ...     tier_3        1
2          742           35000  ...     tier_2        1
3          664          150000  ...     tier_1        1
4          756          100000  ...     tier_1        1
5          721           50000  ...     tier_3        1
6          838           50000  ...     tier_3        1
7          670          150000  ...     tier_3        1
8          752          150000  ...     tier_2        1
9          771           50000  ...     tier_1        1

[10 rows x 6 columns]


In [4]:
# ============================================================
# CELL 4: Augment to 5000+ rows
# ============================================================
from sklearn.utils import resample

target_rows = 5000

# Resample with replacement
df_aug = resample(df, replace=True, n_samples=target_rows, random_state=42)
df_aug = df_aug.copy()
df_aug = df_aug.reset_index(drop=True)

# Add noise so rows aren't exact duplicates
rng = np.random.RandomState(42)
df_aug['cibil_score'] = np.clip(df_aug['cibil_score'].values + rng.randint(-30, 30, target_rows), 300, 900)
df_aug['monthly_income'] = (df_aug['monthly_income'].values * rng.uniform(0.85, 1.15, target_rows)).astype(int)
df_aug['existing_emi'] = (df_aug['existing_emi'].values * rng.uniform(0.8, 1.2, target_rows)).astype(int)
df_aug['age'] = np.clip(df_aug['age'].values + rng.uniform(-2, 2, target_rows), 18, 70).round(2)
ye = df_aug['years_employed'].astype(float).values
df_aug['years_employed'] = np.clip(ye + rng.uniform(-0.5, 0.5, target_rows), 0, None).round(2)

print(f"Augmented rows: {df_aug.shape[0]}")
approved_count = int(df_aug['approved'].sum())
denied_count = int((df_aug['approved'] == 0).sum())
print(f"Approved: {approved_count}, Denied: {denied_count}")

Augmented rows: 5000
Approved: 2261, Denied: 2739


In [5]:
# ============================================================
# CELL 5: Feature Engineering
# ============================================================

# Debt to income ratio
income_vals = df_aug['monthly_income'].values.astype(float)
emi_vals = df_aug['existing_emi'].values.astype(float)
df_aug['debt_to_income'] = np.round(emi_vals / income_vals, 4)

# Monthly surplus (income minus EMI)
df_aug['monthly_surplus'] = (income_vals - emi_vals).astype(int)

# Credit history months (derived from years_employed)
ye_vals = df_aug['years_employed'].astype(float).values
rng2 = np.random.RandomState(99)
df_aug['credit_history_months'] = np.clip((ye_vals * 12 + rng2.randint(0, 24, len(df_aug))), 0, None).astype(int)

print("Feature engineering done. New columns:")
print(df_aug[['debt_to_income', 'monthly_surplus', 'credit_history_months', 'approved']].describe())

Feature engineering done. New columns:
       debt_to_income  monthly_surplus  credit_history_months    approved
count     5000.000000      5000.000000             5000.00000  5000.00000
mean         0.198887     37088.465800               38.41300     0.45220
std          0.123320     31579.703736               40.39804     0.49776
min          0.000000      5319.000000                0.00000     0.00000
25%          0.090125     16646.000000               15.00000     0.00000
50%          0.195100     25656.000000               25.00000     0.00000
75%          0.298475     42853.000000               46.00000     1.00000
max          0.544400    165284.000000              360.00000     1.00000


In [6]:
# ============================================================
# CELL 6: Train / Validation / Test Split (70/15/15)
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

numeric_cols = [
    'cibil_score', 'monthly_income', 'existing_emi', 'age',
    'years_employed', 'debt_to_income', 'monthly_surplus',
    'credit_history_months'
]
categorical_cols = ['employment_type', 'city_tier']

# One-hot encode: employment_type, city_tier
dummies = pd.get_dummies(df_aug[categorical_cols], columns=categorical_cols, dtype=float)
encoded_cols = list(dummies.columns)
feature_cols = numeric_cols + encoded_cols

X = pd.concat([df_aug[numeric_cols], dummies], axis=1)
y = df_aug['approved'].copy()

# First split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
# Second split: 50/50 of temp -> 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f"Train: {X_train.shape[0]}, Validation: {X_val.shape[0]}, Test: {X_test.shape[0]}")
print(f"Features: {len(feature_cols)} ({len(numeric_cols)} numeric + {len(encoded_cols)} one-hot)")
print(f"One-hot columns: {encoded_cols}")

# Normalize numeric features only -- one-hot columns stay 0/1 so SHAP stays readable
scaler = StandardScaler()


def scale_frame(frame, fit=False):
    out = frame.copy()
    values = (
        scaler.fit_transform(frame[numeric_cols])
        if fit
        else scaler.transform(frame[numeric_cols])
    )
    out[numeric_cols] = values
    return out[feature_cols]


X_train_s = scale_frame(X_train, fit=True)
X_val_s = scale_frame(X_val)
X_test_s = scale_frame(X_test)

print("\nScaled training data sample:")
print(X_train_s.head())

Train: 3500, Validation: 750, Test: 750
Features: 15 (8 numeric + 7 one-hot)
One-hot columns: ['employment_type_business', 'employment_type_freelancer', 'employment_type_salaried', 'employment_type_self_employed', 'city_tier_tier_1', 'city_tier_tier_2', 'city_tier_tier_3']

Scaled training data sample:
      cibil_score  monthly_income  ...  city_tier_tier_2  city_tier_tier_3
1840     0.458872       -0.897256  ...               0.0               0.0
2115     1.070974        1.682365  ...               0.0               0.0
4437     0.633758        0.963508  ...               0.0               0.0
1146     1.379940        0.929660  ...               0.0               0.0
2486     0.359770        1.589335  ...               1.0               0.0

[5 rows x 15 columns]


In [7]:
# ============================================================
# CELL 7: Train XGBoost with Grid Search
# ============================================================
from sklearn.model_selection import GridSearchCV
import xgboost as xgb

xgb_params = {
    'max_depth': [4, 5, 6],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200]
}

xgb_model = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

xgb_grid = GridSearchCV(
    xgb_model, xgb_params,
    cv=3, scoring='accuracy', verbose=1
)
xgb_grid.fit(X_train_s, y_train)

print(f"\nBest XGBoost params: {xgb_grid.best_params_}")
print(f"Best XGBoost CV accuracy: {xgb_grid.best_score_:.4f}")
print(f"Validation accuracy: {xgb_grid.best_estimator_.score(X_val_s, y_val):.4f}")

Fitting 3 folds for each of 12 candidates, totalling 36 fits

Best XGBoost params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
Best XGBoost CV accuracy: 0.9829
Validation accuracy: 0.9867


In [8]:
# ============================================================
# CELL 8: Train LightGBM with Grid Search
# ============================================================
import lightgbm as lgb

lgb_params = {
    'num_leaves': [15, 31],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200]
}

lgb_model = lgb.LGBMClassifier(random_state=42, verbose=-1)

lgb_grid = GridSearchCV(
    lgb_model, lgb_params,
    cv=3, scoring='accuracy', verbose=1
)
lgb_grid.fit(X_train_s, y_train)

print(f"\nBest LightGBM params: {lgb_grid.best_params_}")
print(f"Best LightGBM CV accuracy: {lgb_grid.best_score_:.4f}")
print(f"Validation accuracy: {lgb_grid.best_estimator_.score(X_val_s, y_val):.4f}")

Fitting 3 folds for each of 8 candidates, totalling 24 fits

Best LightGBM params: {'learning_rate': 0.1, 'n_estimators': 200, 'num_leaves': 15}
Best LightGBM CV accuracy: 0.9877
Validation accuracy: 0.9867


In [9]:
# ============================================================
# CELL 9: Ensemble Model (Soft Voting)
# ============================================================
from sklearn.ensemble import VotingClassifier

ensemble = VotingClassifier(
    estimators=[
        ('xgb', xgb_grid.best_estimator_),
        ('lgb', lgb_grid.best_estimator_)
    ],
    voting='soft'
)
ensemble.fit(X_train_s, y_train)

print(f"Ensemble validation accuracy: {ensemble.score(X_val_s, y_val):.4f}")

Ensemble validation accuracy: 0.9867


In [10]:
# ============================================================
# CELL 10: Evaluate All Models on Test Set
# ============================================================
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

models = {
    'XGBoost': xgb_grid.best_estimator_,
    'LightGBM': lgb_grid.best_estimator_,
    'Ensemble': ensemble
}

for name, model in models.items():
    y_pred = model.predict(X_test_s)
    y_prob = model.predict_proba(X_test_s)[:, 1]
    print(f"\n{'=' * 40}")
    print(f"{name} Results:")
    print(f"{'=' * 40}")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
    print(f"AUC-ROC:   {roc_auc_score(y_test, y_prob):.4f}")


XGBoost Results:
Accuracy:  0.9920
Precision: 0.9935
Recall:    0.9871
F1 Score:  0.9903
AUC-ROC:   0.9990

LightGBM Results:
Accuracy:  0.9947
Precision: 0.9935
Recall:    0.9935
F1 Score:  0.9935
AUC-ROC:   0.9994

Ensemble Results:
Accuracy:  0.9920
Precision: 0.9903
Recall:    0.9903
F1 Score:  0.9903
AUC-ROC:   0.9991


In [11]:
# ============================================================
# CELL 11: SHAP Explainability
# ============================================================
import shutil

import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
import shap

artifacts_dir = os.path.join(BASE_DIR, "ml", "artifacts")
os.makedirs(artifacts_dir, exist_ok=True)


def binary_shap_values(explainer, frame):
    """TreeExplainer output for the positive class, across model types."""
    values = explainer.shap_values(frame)
    if isinstance(values, list):          # some LightGBM builds return [class0, class1]
        values = values[1]
    values = np.asarray(values)
    if values.ndim == 3:                  # (rows, features, classes)
        values = values[:, :, 1]
    return values


# TreeExplainer for BOTH models
xgb_explainer = shap.TreeExplainer(xgb_grid.best_estimator_)
lgb_explainer = shap.TreeExplainer(lgb_grid.best_estimator_)
explainers = {'xgb': xgb_explainer, 'lgb': lgb_explainer}

for name, explainer in explainers.items():
    sv = binary_shap_values(explainer, X_test_s)
    shap.summary_plot(sv, X_test_s, feature_names=feature_cols, show=False)
    plt.tight_layout()
    out_path = os.path.join(artifacts_dir, f"shap_summary_{name}.png")
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"{name.upper()} SHAP summary -> {out_path}")

# Default summary image keeps its original name, sourced from XGBoost
shutil.copyfile(
    os.path.join(artifacts_dir, "shap_summary_xgb.png"),
    os.path.join(artifacts_dir, "shap_summary.png"),
)

shap_values = binary_shap_values(xgb_explainer, X_test_s)
print("SHAP summary plot saved to ml/artifacts/shap_summary.png")

XGB SHAP summary -> /Users/krishnareddy/ my Projects/credlytic/ml/artifacts/shap_summary_xgb.png
LGB SHAP summary -> /Users/krishnareddy/ my Projects/credlytic/ml/artifacts/shap_summary_lgb.png
SHAP summary plot saved to ml/artifacts/shap_summary.png


In [12]:
# ============================================================
# CELL 12: Human-Readable Explanation for a Single Prediction
# ============================================================

UNITS = {
    'cibil_score': '{:.0f}',
    'monthly_income': 'Rs {:,.0f}',
    'existing_emi': 'Rs {:,.0f}',
    'monthly_surplus': 'Rs {:,.0f}',
    'age': '{:.0f} years',
    'years_employed': '{:.1f} years',
    'debt_to_income': '{:.1%}',
    'credit_history_months': '{:.0f} months',
}


def format_value(feature, value):
    if feature in UNITS:
        return UNITS[feature].format(value)
    return 'yes' if value >= 0.5 else 'no'


def explain_prediction(model, shap_explainer, row_scaled, row_raw, cols):
    """Human-readable explanation for one prediction, in real-world units."""
    prob = model.predict_proba(row_scaled)[0]
    sv = binary_shap_values(shap_explainer, row_scaled)[0]
    raw_vals = row_raw[cols].values[0]

    print(f"Approval probability: {prob[1]:.2%}")
    print("\nTop 5 factors:")

    ranked = sorted(zip(cols, raw_vals, sv), key=lambda x: abs(x[2]), reverse=True)
    for feat, val, s in ranked[:5]:
        direction = "increasing" if s > 0 else "decreasing"
        print(f"  -> Your {feat} of {format_value(feat, val)} is {direction} "
              f"your probability by {abs(s):.3f} points")


# Test with first row from test set -- scaled row for the model, raw row for display
explain_prediction(
    xgb_grid.best_estimator_,
    xgb_explainer,
    X_test_s.iloc[[0]],
    X_test.iloc[[0]],
    feature_cols,
)

Approval probability: 97.29%

Top 5 factors:
  -> Your age of 66 years is increasing your probability by 1.466 points
  -> Your years_employed of 13.3 years is increasing your probability by 1.149 points
  -> Your cibil_score of 669 is increasing your probability by 0.833 points
  -> Your city_tier_tier_2 of yes is decreasing your probability by 0.434 points
  -> Your existing_emi of Rs 3,701 is increasing your probability by 0.384 points


In [13]:
# ============================================================
# CELL 13: Save Model Artifacts
# ============================================================
import joblib
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
artifacts_dir = os.path.join(BASE_DIR, "ml", "artifacts")
os.makedirs(artifacts_dir, exist_ok=True)

# Save timestamped versions
joblib.dump(xgb_grid.best_estimator_, os.path.join(artifacts_dir, f"xgb_model_{timestamp}.joblib"))
joblib.dump(lgb_grid.best_estimator_, os.path.join(artifacts_dir, f"lgb_model_{timestamp}.joblib"))
joblib.dump(ensemble, os.path.join(artifacts_dir, f"ensemble_model_{timestamp}.joblib"))
joblib.dump(scaler, os.path.join(artifacts_dir, f"scaler_{timestamp}.joblib"))

# Save "latest" versions for the API to load
joblib.dump(ensemble, os.path.join(artifacts_dir, "ensemble_model_latest.joblib"))
joblib.dump(scaler, os.path.join(artifacts_dir, "scaler_latest.joblib"))
joblib.dump(xgb_grid.best_estimator_, os.path.join(artifacts_dir, "xgb_model_latest.joblib"))
joblib.dump(lgb_grid.best_estimator_, os.path.join(artifacts_dir, "lgb_model_latest.joblib"))

# Feature contract the prediction endpoint needs to rebuild a row in the right order
schema = {
    "feature_cols": feature_cols,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "encoded_cols": encoded_cols,
    "trained_at": timestamp,
}
joblib.dump(schema, os.path.join(artifacts_dir, f"feature_schema_{timestamp}.joblib"))
joblib.dump(schema, os.path.join(artifacts_dir, "feature_schema_latest.joblib"))

# Save the augmented dataset
data_out = os.path.join(BASE_DIR, "data", "datasets", "credit_augmented.csv")
df_aug.to_csv(data_out, index=False)

print(f"Models saved to: {artifacts_dir}")
print(f"Dataset saved to: {data_out}")
print(f"Timestamp: {timestamp}")
print(f"Feature schema: {len(feature_cols)} features")
print(f"\nFiles saved:")
for f in sorted(os.listdir(artifacts_dir)):
    if f.endswith('.joblib'):
        print(f"  {f}")

Models saved to: /Users/krishnareddy/ my Projects/credlytic/ml/artifacts
Dataset saved to: /Users/krishnareddy/ my Projects/credlytic/data/datasets/credit_augmented.csv
Timestamp: 20261007_230150
Feature schema: 15 features

Files saved:
  ensemble_model_20260930_212937.joblib
  ensemble_model_20260930_213512.joblib
  ensemble_model_20261007_230150.joblib
  ensemble_model_latest.joblib
  feature_schema_20261007_230150.joblib
  feature_schema_latest.joblib
  lgb_model_20260930_212937.joblib
  lgb_model_20260930_213512.joblib
  lgb_model_20261007_230150.joblib
  lgb_model_latest.joblib
  scaler_20260930_212937.joblib
  scaler_20260930_213512.joblib
  scaler_20261007_230150.joblib
  scaler_latest.joblib
  xgb_model_20260930_212937.joblib
  xgb_model_20260930_213512.joblib
  xgb_model_20261007_230150.joblib
  xgb_model_latest.joblib
